# Psychoacoustics: the reference sounds

Each psychoacoustic unit is defined by a reference sound. This notebook computes them with `pytta.psychoacoustics`; the results should be close to 1 in each case. The signals are sound pressures in pascal.

In [1]:
import numpy as np
import pytta

fs = 48000
t = np.arange(2 * fs) / fs
p = lambda level: np.sqrt(2) * 2e-5 * 10 ** (level / 20)    # amplitude of a sine in Pa
P = pytta.psychoacoustics

## Loudness

A 1 kHz tone at 40 dB in a free field is 1 sone, or a loudness level of 40 phon (ISO 532-1).

In [2]:
tone = pytta.SignalObj(p(40) * np.sin(2 * np.pi * 1000 * t), 'time', fs)
result = P.loudness_zwicker(tone, field='free')
print(f"N = {result['N']:.3f} sone, LN = {result['LN']:.2f} phon")

N = 1.000 sone, LN = 40.00 phon


## Sharpness

Narrow-band noise one critical band wide at 1 kHz and 60 dB is 1 acum (DIN 45692).

In [3]:
rng = np.random.default_rng(1)
X = np.fft.rfft(rng.standard_normal(t.size))
f = np.fft.rfftfreq(t.size, 1 / fs)
X[(f < 920) | (f > 1080)] = 0
noise = np.fft.irfft(X, t.size)
noise *= 2e-5 * 10 ** (60 / 20) / np.sqrt(np.mean(noise ** 2))
print(f"S = {P.sharpness_din45692(pytta.SignalObj(noise, 'time', fs))['S']:.3f} acum")

S = 1.020 acum


## Roughness

A 1 kHz tone at 60 dB, fully amplitude modulated at 70 Hz, is 1 asper (Daniel and Weber 1997).

In [4]:
am = (1 + np.cos(2 * np.pi * 70 * t)) * np.sin(2 * np.pi * 1000 * t)
am *= 2e-5 * 10 ** (60 / 20) / np.sqrt(np.mean(am ** 2))
r = P.roughness_daniel1997(pytta.SignalObj(am, 'time', fs), time_skip=0.5)
print(f"R = {r['Rmean']:.3f} asper")

R = 1.000 asper


## Fluctuation strength

A 1 kHz tone at 60 dB, fully amplitude modulated at 4 Hz, is 1 vacil (Osses et al. 2016).

In [5]:
fm = (1 + np.cos(2 * np.pi * 4 * t)) * np.sin(2 * np.pi * 1000 * t)
fm *= 2e-5 * 10 ** (60 / 20) / np.sqrt(np.mean(fm ** 2))
F = P.fluctuation_strength_osses2016(pytta.SignalObj(fm, 'time', fs), time_skip=0.5)
print(f"F = {F['FSmean']:.3f} vacil")

F = 0.941 vacil
